# Sustainability Experiment Analysis
## Baseline vs Data Reuse — Energy, Throughput, Duration

In [2]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

plt.rcParams.update({
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.dpi': 150,
    'axes.grid': True,
    'grid.alpha': 0.3
})

## 1. Load SMA Data

In [3]:
# ── Update these paths to your local data folders ──────────────────────────
BASE_DIR = Path("~/Desktop/paper_experiments").expanduser()

# Map experiment name -> SMA folder path
# Replace folder names with your actual folder names
EXPERIMENTS = {
    # Baseline
    "Baseline U2": BASE_DIR / "baseline_sequential" / "nextflow_rnaseq_partial_resume_2026_05_28_06_56_27_df3720c9",
    "Baseline U4": BASE_DIR / "baseline_sequential" / "nextflow_rnaseq_partial_resume_2026_05_28_08_23_15_380543a9",

    # Data Reuse U2
    "Reuse U2 10%": BASE_DIR / "reuse_sequential" / "nextflow_rnaseq_partial_resume_2026_05_27_10_34_11_31e60571",
    "Reuse U2 20%": BASE_DIR / "reuse_sequential" / "nextflow_rnaseq_partial_resume_2026_05_27_12_07_32_3d7b68ca",
    "Reuse U2 50%": BASE_DIR / "reuse_sequential" / "nextflow_rnaseq_partial_resume_2026_05_27_13_53_11_2e5a48b1",

    # Data Reuse U4
    "Reuse U4 10%": BASE_DIR / "reuse_sequential" / "nextflow_rnaseq_partial_resume_2026_05_27_15_19_12_fb52d4d4",
    "Reuse U4 20%": BASE_DIR / "reuse_sequential" / "nextflow_rnaseq_partial_resume_2026_05_27_18_29_51_f387afe0",
    "Reuse U4 50%": BASE_DIR / "reuse_sequential" / "nextflow_rnaseq_partial_resume_2026_05_27_22_10_43_9bb8ec96",
}

def load_run_json(folder):
    run_file = Path(folder) / "run.json"
    if run_file.exists():
        return json.loads(run_file.read_text())
    return None

def load_csv(folder, name):
    path = Path(folder) / "data" / name
    if path.exists():
        return pd.read_csv(path)
    return None

# Load all experiment metadata
run_data = {}
for name, folder in EXPERIMENTS.items():
    rj = load_run_json(folder)
    if rj:
        run_data[name] = rj
        print(f"✅ {name}: duration={rj.get('treatment_duration', 'N/A'):.0f}s")
    else:
        print(f"❌ {name}: run.json not found at {folder}")

print(f"\nLoaded {len(run_data)} experiments")

✅ Baseline U2: duration=5147s
✅ Baseline U4: duration=10373s
✅ Reuse U2 10%: duration=4937s
✅ Reuse U2 20%: duration=4914s
✅ Reuse U2 50%: duration=3952s
✅ Reuse U4 10%: duration=9653s
✅ Reuse U4 20%: duration=9077s
✅ Reuse U4 50%: duration=6673s

Loaded 8 experiments


## 2. Load Energy Data (Kepler)

In [7]:
df = pd.read_csv(folder / "data" / "kepler_node_cpu_power_raw.csv")
treatment = df[df['treatment'] == 'Treatment']
print(treatment['kepler_node_cpu_power_raw'].max() - treatment['kepler_node_cpu_power_raw'].min())

192009.59109499998


In [4]:
def get_total_energy_joules(folder):
    """Get total energy from kepler_node_cpu_power_raw.csv (counter metric - take max - min)"""
    df = load_csv(folder, "kepler_node_cpu_power_raw.csv")
    if df is not None and not df.empty:
        # Counter metric: total = max - min over the measurement window
        return df['value'].max() - df['value'].min()
    
    # Fallback: use kepler_all_pod_cpu_power_watts.csv (aggregate watts, integrate over time)
    df = load_csv(folder, "kepler_all_pod_cpu_power_watts.csv")
    if df is not None and not df.empty:
        df = df.sort_values('timestamp') if 'timestamp' in df.columns else df
        # Integrate watts over time (step=10s)
        return df['value'].sum() * 10  # joules
    return None

def get_treatment_duration(folder):
    rj = load_run_json(folder)
    if rj:
        return rj.get('treatment_duration', None)
    return None

# Collect energy and duration for all experiments
results = []
for name, folder in EXPERIMENTS.items():
    energy = get_total_energy_joules(folder)
    duration = get_treatment_duration(folder)
    rj = load_run_json(folder)
    results.append({
        'experiment': name,
        'energy_joules': energy,
        'duration_s': duration,
        'duration_min': duration / 60 if duration else None,
    })

df_results = pd.DataFrame(results)
print(df_results[['experiment', 'energy_joules', 'duration_min']].to_string(index=False))

KeyError: 'value'

## 3. Load Nextflow Trace Files

In [ ]:
TRACE_DIR = BASE_DIR / "reuse_sequential" / "traces"
BASELINE_TRACE_DIR = BASE_DIR / "baseline_sequential" / "traces"

def load_trace(trace_path):
    """Load Nextflow trace.txt and return DataFrame"""
    if Path(trace_path).exists():
        df = pd.read_csv(trace_path, sep='\t')
        return df
    return None

# Load all traces
traces = {}
for run_folder in TRACE_DIR.glob("*/trace.txt"):
    run_id = run_folder.parent.name
    traces[run_id] = load_trace(run_folder)
    
for run_folder in BASELINE_TRACE_DIR.glob("*/trace.txt"):
    run_id = run_folder.parent.name
    traces[run_id] = load_trace(run_folder)

print(f"Loaded {len(traces)} trace files")
for k in list(traces.keys())[:5]:
    print(f"  {k}: {len(traces[k])} tasks" if traces[k] is not None else f"  {k}: empty")

## 4. Plot 1 — Total Energy Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# Group experiments
groups = {
    'U2': {
        'Baseline':  df_results[df_results.experiment == 'Baseline U2']['energy_joules'].values[0],
        '10% Reuse': df_results[df_results.experiment == 'Reuse U2 10%']['energy_joules'].values[0],
        '20% Reuse': df_results[df_results.experiment == 'Reuse U2 20%']['energy_joules'].values[0],
        '50% Reuse': df_results[df_results.experiment == 'Reuse U2 50%']['energy_joules'].values[0],
    },
    'U4': {
        'Baseline':  df_results[df_results.experiment == 'Baseline U4']['energy_joules'].values[0],
        '10% Reuse': df_results[df_results.experiment == 'Reuse U4 10%']['energy_joules'].values[0],
        '20% Reuse': df_results[df_results.experiment == 'Reuse U4 20%']['energy_joules'].values[0],
        '50% Reuse': df_results[df_results.experiment == 'Reuse U4 50%']['energy_joules'].values[0],
    }
}

x = np.arange(4)
width = 0.35
colors_u2 = ['#1f77b4', '#aec7e8', '#6baed6', '#2171b5']
colors_u4 = ['#d62728', '#fc9272', '#ef6548', '#a50f15']

bars1 = ax.bar(x - width/2, list(groups['U2'].values()), width,
               label='U2', color=colors_u2, edgecolor='black', linewidth=0.5)
bars2 = ax.bar(x + width/2, list(groups['U4'].values()), width,
               label='U4', color=colors_u4, edgecolor='black', linewidth=0.5)

ax.set_xlabel('Scenario')
ax.set_ylabel('Total Energy (Joules)')
ax.set_title('Total Energy Consumption: Baseline vs Data Reuse\n(U2 and U4 Users)')
ax.set_xticks(x)
ax.set_xticklabels(['Baseline', '10% Reuse', '20% Reuse', '50% Reuse'])

# Add value labels on bars
for bar in bars1:
    h = bar.get_height()
    if h and not np.isnan(h):
        ax.text(bar.get_x() + bar.get_width()/2., h + 0.01*h,
                f'{h:.0f}J', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    h = bar.get_height()
    if h and not np.isnan(h):
        ax.text(bar.get_x() + bar.get_width()/2., h + 0.01*h,
                f'{h:.0f}J', ha='center', va='bottom', fontsize=9)

ax.legend(title='User Count')
plt.tight_layout()
plt.savefig('energy_comparison.pdf', bbox_inches='tight')
plt.savefig('energy_comparison.png', bbox_inches='tight')
plt.show()
print("Saved: energy_comparison.pdf")

## 5. Plot 2 — Pipeline Duration

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

groups_dur = {
    'U2': {
        'Baseline':  df_results[df_results.experiment == 'Baseline U2']['duration_min'].values[0],
        '10% Reuse': df_results[df_results.experiment == 'Reuse U2 10%']['duration_min'].values[0],
        '20% Reuse': df_results[df_results.experiment == 'Reuse U2 20%']['duration_min'].values[0],
        '50% Reuse': df_results[df_results.experiment == 'Reuse U2 50%']['duration_min'].values[0],
    },
    'U4': {
        'Baseline':  df_results[df_results.experiment == 'Baseline U4']['duration_min'].values[0],
        '10% Reuse': df_results[df_results.experiment == 'Reuse U4 10%']['duration_min'].values[0],
        '20% Reuse': df_results[df_results.experiment == 'Reuse U4 20%']['duration_min'].values[0],
        '50% Reuse': df_results[df_results.experiment == 'Reuse U4 50%']['duration_min'].values[0],
    }
}

bars1 = ax.bar(x - width/2, list(groups_dur['U2'].values()), width,
               label='U2', color=colors_u2, edgecolor='black', linewidth=0.5)
bars2 = ax.bar(x + width/2, list(groups_dur['U4'].values()), width,
               label='U4', color=colors_u4, edgecolor='black', linewidth=0.5)

ax.set_xlabel('Scenario')
ax.set_ylabel('Total Duration (minutes)')
ax.set_title('Total Pipeline Duration: Baseline vs Data Reuse\n(U2 and U4 Users)')
ax.set_xticks(x)
ax.set_xticklabels(['Baseline', '10% Reuse', '20% Reuse', '50% Reuse'])

for bar in bars1:
    h = bar.get_height()
    if h and not np.isnan(h):
        ax.text(bar.get_x() + bar.get_width()/2., h + 0.01*h,
                f'{h:.1f}m', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    h = bar.get_height()
    if h and not np.isnan(h):
        ax.text(bar.get_x() + bar.get_width()/2., h + 0.01*h,
                f'{h:.1f}m', ha='center', va='bottom', fontsize=9)

ax.legend(title='User Count')
plt.tight_layout()
plt.savefig('duration_comparison.pdf', bbox_inches='tight')
plt.savefig('duration_comparison.png', bbox_inches='tight')
plt.show()
print("Saved: duration_comparison.pdf")

## 6. Plot 3 — System Throughput (tasks/min)

In [ ]:
# Throughput = total tasks completed / total duration
# For sequential runs: total tasks = num_users * 548

throughput = {}
user_counts = {'U2': 2, 'U4': 4}
tasks_per_user = 548

for exp_name, folder in EXPERIMENTS.items():
    rj = load_run_json(folder)
    if rj and rj.get('treatment_duration'):
        dur_min = rj['treatment_duration'] / 60
        u = 2 if 'U2' in exp_name else 4
        total_tasks = u * tasks_per_user
        throughput[exp_name] = total_tasks / dur_min

fig, ax = plt.subplots(figsize=(12, 6))

groups_tp = {
    'U2': {
        'Baseline':  throughput.get('Baseline U2', 0),
        '10% Reuse': throughput.get('Reuse U2 10%', 0),
        '20% Reuse': throughput.get('Reuse U2 20%', 0),
        '50% Reuse': throughput.get('Reuse U2 50%', 0),
    },
    'U4': {
        'Baseline':  throughput.get('Baseline U4', 0),
        '10% Reuse': throughput.get('Reuse U4 10%', 0),
        '20% Reuse': throughput.get('Reuse U4 20%', 0),
        '50% Reuse': throughput.get('Reuse U4 50%', 0),
    }
}

bars1 = ax.bar(x - width/2, list(groups_tp['U2'].values()), width,
               label='U2', color=colors_u2, edgecolor='black', linewidth=0.5)
bars2 = ax.bar(x + width/2, list(groups_tp['U4'].values()), width,
               label='U4', color=colors_u4, edgecolor='black', linewidth=0.5)

ax.set_xlabel('Scenario')
ax.set_ylabel('Throughput (tasks / minute)')
ax.set_title('System Throughput: Baseline vs Data Reuse\n(U2 and U4 Users)')
ax.set_xticks(x)
ax.set_xticklabels(['Baseline', '10% Reuse', '20% Reuse', '50% Reuse'])

for bar in bars1:
    h = bar.get_height()
    if h and not np.isnan(h):
        ax.text(bar.get_x() + bar.get_width()/2., h + 0.01*h,
                f'{h:.1f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    h = bar.get_height()
    if h and not np.isnan(h):
        ax.text(bar.get_x() + bar.get_width()/2., h + 0.01*h,
                f'{h:.1f}', ha='center', va='bottom', fontsize=9)

ax.legend(title='User Count')
plt.tight_layout()
plt.savefig('throughput_comparison.pdf', bbox_inches='tight')
plt.savefig('throughput_comparison.png', bbox_inches='tight')
plt.show()
print("Saved: throughput_comparison.pdf")

## 7. Plot 4 — Energy Savings % vs Reuse %

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

reuse_pcts = [0, 10, 20, 50]

baseline_u2 = df_results[df_results.experiment == 'Baseline U2']['energy_joules'].values[0]
baseline_u4 = df_results[df_results.experiment == 'Baseline U4']['energy_joules'].values[0]

savings_u2 = [0] + [
    (baseline_u2 - df_results[df_results.experiment == f'Reuse U2 {p}%']['energy_joules'].values[0]) / baseline_u2 * 100
    for p in [10, 20, 50]
]
savings_u4 = [0] + [
    (baseline_u4 - df_results[df_results.experiment == f'Reuse U4 {p}%']['energy_joules'].values[0]) / baseline_u4 * 100
    for p in [10, 20, 50]
]

ax.plot(reuse_pcts, savings_u2, 'o-', color='#1f77b4', linewidth=2, markersize=8, label='U2')
ax.plot(reuse_pcts, savings_u4, 's-', color='#d62728', linewidth=2, markersize=8, label='U4')

for i, (p, s) in enumerate(zip(reuse_pcts, savings_u2)):
    ax.annotate(f'{s:.1f}%', (p, s), textcoords='offset points', xytext=(0, 10), ha='center', fontsize=10, color='#1f77b4')
for i, (p, s) in enumerate(zip(reuse_pcts, savings_u4)):
    ax.annotate(f'{s:.1f}%', (p, s), textcoords='offset points', xytext=(0, -18), ha='center', fontsize=10, color='#d62728')

ax.set_xlabel('Cache Reuse Percentage (%)')
ax.set_ylabel('Energy Savings vs Baseline (%)')
ax.set_title('Energy Savings as a Function of Data Reuse Percentage')
ax.set_xticks(reuse_pcts)
ax.legend(title='User Count')
ax.axhline(y=0, color='black', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig('energy_savings_vs_reuse.pdf', bbox_inches='tight')
plt.savefig('energy_savings_vs_reuse.png', bbox_inches='tight')
plt.show()
print("Saved: energy_savings_vs_reuse.pdf")

## 8. Summary Table

In [ ]:
summary = df_results.copy()
summary['throughput_tasks_min'] = summary.apply(
    lambda r: (2 if 'U2' in r['experiment'] else 4) * tasks_per_user / r['duration_min']
    if r['duration_min'] else None, axis=1
)

print(summary[['experiment', 'energy_joules', 'duration_min', 'throughput_tasks_min']].to_string(index=False))
summary.to_csv('experiment_summary.csv', index=False)
print("\nSaved: experiment_summary.csv")